# GWAS and xQTL integration mini-protocol
Integrate molecular-QTL results with GWAS using enrichment, colocalization, TWAS, cTWAS, quantile TWAS, or INTACT.

#### Miniprotocol Timing
This is the total duration for the selected route; module-specific timings appear on their respective pages.
Timing: TBD

## Overview
This mini-protocol presents complementary routes for connecting molecular-QTL and GWAS evidence. Steps 1–2 call [`SuSiE_enloc.ipynb`](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html), steps 3–7 call [`twas_ctwas.ipynb`](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html), and step 8 calls [`intact.ipynb`](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/intact.html).
Enrichment, colocalization, standard TWAS, cTWAS, quantile TWAS, and INTACT answer different questions and are not one mandatory chain. Only cTWAS steps 4–6 must be run sequentially.

## Steps
Choose a route before running commands; the commands are not one mandatory chain.
| Analysis goal | Commands to run, in order | Inputs |
|---|---|---|
| Estimate global xQTL–GWAS enrichment | 1 | `input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv`; `input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv`; fine-mapping objects referenced by those tables |
| Test pairwise colocalization | 2 | The same GWAS and xQTL fine-mapping metadata as step 1 |
| Run standard TWAS and MR | 3 | `input/twas/protocol_example.twas.gwas_meta.tsv`; `input/twas/protocol_example.twas.xqtl_meta.tsv`; `input/ld_reference/protocol_example.ld_meta_file.tsv`; `input/twas/protocol_example.twas.LD_blocks.chr22.bed`; `input/twas/protocol_example.twas.data_type_table.txt` |
| Fine-map genes and SNPs jointly with cTWAS | 4 → 5 → 6 | GWAS, xQTL, LD-reference, and region metadata used by the TWAS route |
| Run quantile TWAS | 7 | The TWAS inputs, with quantile-specific molecular-QTL weights |
| Combine PTWAS and fastenloc evidence | 8 | `input/twas/protocol_example.ptwas.output`; `input/twas/protocol_example.fastenloc.gene.out` |

Run only the commands for the selected route. Steps 4–6 are one chained cTWAS analysis; the other numbered steps are independent alternatives.

### 1. [Estimate global xQTL–GWAS enrichment](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html)

**What it does:** Estimates whether fine-mapped xQTL signals are enriched among GWAS signals across many regions and molecular contexts.

**Timing**: TBD

In [ ]:
sos run pipeline/SuSiE_enloc.ipynb xqtl_gwas_enrichment \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl-finemapping-obj preset_variants_result susie_result_trimmed \
    --xqtl-varname-obj preset_variants_result variant_names \
    --gwas-finemapping-obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas-varname-obj  AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl-region-obj   region_info grange \
    --qtl-path input/susie_enloc_data \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --cwd output/xqtl_gwas_enrichment

### 2. [Test pairwise colocalization](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/SuSiE_enloc.html)

**What it does:** Tests whether an xQTL and GWAS association in the same region are consistent with a shared causal variant.

**Timing**: TBD

In [ ]:
sos run pipeline/SuSiE_enloc.ipynb susie_coloc \
    --gwas-meta-data input/susie_enloc_data/protocol_example.enloc.gwas_meta.tsv \
    --xqtl-meta-data input/susie_enloc_data/protocol_example.enloc.xqtl_meta.tsv \
    --xqtl-finemapping-obj preset_variants_result susie_result_trimmed \
    --xqtl-varname-obj preset_variants_result variant_names \
    --gwas-finemapping-obj AD_Bellenguez_2022 RSS_QC_RAISS_imputed susie_result_trimmed \
    --gwas-varname-obj  AD_Bellenguez_2022 RSS_QC_RAISS_imputed variant_names \
    --xqtl-region-obj   region_info grange \
    --qtl-path input/susie_enloc_data \
    --gwas-path input/susie_enloc_data \
    --context-meta input/susie_enloc_data/protocol_example.enloc.context_meta.tsv \
    --ld-meta-file-path input/ld_reference/protocol_example.ld_meta_file.tsv \
    --skip-enrich \
    --cwd output/susie_coloc

### 3. [Run standard TWAS and MR](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html)

**What it does:** Tests genetically predicted molecular traits for association with the GWAS trait and reports regional Mendelian-randomization results.

**Timing**: TBD

In [ ]:
sos run pipeline/twas_ctwas.ipynb twas \
    --cwd output --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --ld_meta_data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --ld_reference_sample_size 17000 \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --xqtl_type_table input/twas/protocol_example.twas.data_type_table.txt \
    --rsq_pval_cutoff 0.05 --rsq_cutoff 0.01 \
    --region-name chr22_10000000_19000000

### 4. [Assemble cTWAS region data](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html)

**What it does:** Harmonizes GWAS statistics, xQTL weights, and LD for the regions that will enter cTWAS fine-mapping.

**Timing**: TBD

In [ ]:
sos run pipeline/twas_ctwas.ipynb ctwas \
    --cwd output --name protocol_example \
    --thin 1 --prior_var_structure shared_all \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --ld_meta_data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --twas_weight_cutoff 0 \
    --region-name chr22_10000000_19000000

### 5. [Estimate cTWAS global parameters](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html)

**What it does:** Reuses the assembled inputs from step 4 to estimate the prior parameters required by cTWAS.

**Timing**: TBD

In [ ]:
sos run pipeline/twas_ctwas.ipynb ctwas \
    --run_param_est --skip_assembly --thin 1 \
    --prior_var_structure shared_all \
    --cwd output --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --ld_meta_data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed

### 6. [Fine-map genes and SNPs with cTWAS](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html)

**What it does:** Uses the assembled inputs and estimated parameters to calculate joint posterior inclusion probabilities for genes and SNPs.

**Timing**: TBD

In [ ]:
sos run pipeline/twas_ctwas.ipynb ctwas \
    --run_finemapping --skip_assembly \
    --prior_var_structure shared_all \
    --cwd output --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --ld_meta_data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --region-name chr22_10000000_19000000

### 7. [Run quantile TWAS](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/twas_ctwas.html)

**What it does:** Runs the quantile-TWAS route for molecular-trait effects that may differ across the phenotype distribution.

**Timing**: TBD

In [ ]:
sos run pipeline/twas_ctwas.ipynb quantile_twas \
    --cwd output --name protocol_example \
    --gwas_meta_data input/twas/protocol_example.twas.gwas_meta.tsv \
    --xqtl_meta_data input/twas/protocol_example.twas.xqtl_meta.tsv \
    --ld_meta_data input/ld_reference/protocol_example.ld_meta_file.tsv \
    --ld_reference_sample_size 17000 \
    --regions input/twas/protocol_example.twas.LD_blocks.chr22.bed \
    --xqtl_type_table input/twas/protocol_example.twas.data_type_table.txt \
    --region-name chr22_10000000_19000000

### 8. [Combine PTWAS and colocalization with INTACT](https://statfungen.github.io/xqtl-protocol/code/pecotmr_integration/intact.html)

**What it does:** Combines PTWAS z-scores with fastenloc colocalization probabilities to prioritize genes supported by both association and colocalization evidence.

**Timing**: TBD

In [ ]:
sos run pipeline/intact.ipynb intact \
    --fastenloc-file input/twas/protocol_example.fastenloc.gene.out \
    --ptwas-file input/twas/protocol_example.ptwas.output \
    --tissue DLPFC \
    --cwd output/intact

## Output Files

| Step | Relative path | Contents |
|---:|---|---|
| 1 | `output/xqtl_gwas_enrichment/<name>.<context>.enrichment.rds` | Global enrichment estimate for each GWAS–xQTL-context pair |
| 2 | `output/susie_coloc/susie_coloc/<name>.<context>@<gene>.coloc.rds` | Regional SuSiE colocalization probabilities |
| 3 | `output/twas/twas/<name>.<region_id>.<gene>.twas.rds` | Per-gene TWAS results |
| 3 | `output/twas/twas/<name>.<region_id>.twas.tsv.gz` | Regional TWAS summary |
| 3 | `output/twas/twas/<name>.<region_id>.mr_result.tsv.gz` | Regional Mendelian-randomization summary |
| 4 | `output/ctwas/<name>.ctwas_inputs.rds` | Assembled cTWAS inputs |
| 5 | `output/ctwas/<name>.ctwas_est.rds` | Estimated cTWAS prior parameters |
| 6 | `output/ctwas/<name>.ctwas_finemap.rds` | Joint gene and SNP fine-mapping results |
| 7 | `output/quantile_twas/<name>.*` | Quantile-TWAS results, with stems determined by the analysis regions |
| 8 | `output/intact/DLPFC.INTACT.rds` | Gene-level INTACT posterior probabilities |

Some workflows derive context, gene, or region identifiers from metadata, so their output contains one file per analysis unit.

## Anticipated Results

Enrichment evaluates global sharing between molecular-QTL and GWAS signals, whereas colocalization evaluates individual regions. TWAS prioritizes traits whose genetically predicted levels associate with the GWAS outcome; cTWAS jointly fine-maps genes and SNPs; INTACT prioritizes genes supported by both PTWAS and colocalization.

These analyses provide complementary evidence, not interchangeable proof of causality. Interpret prioritized genes together with fine-mapping quality, LD, prediction performance, colocalization support, tissue relevance, and sensitivity analyses.

## Command interface

In [ ]:
sos run pipeline/SuSiE_enloc.ipynb -h

In [ ]:
sos run pipeline/twas_ctwas.ipynb -h

In [ ]:
sos run pipeline/intact.ipynb -h